# 06 — Data Inventory: football-data.co.uk seasons 2024/25 and 2025/26

What do `raw_data/2425.csv` and `raw_data/2526.csv` contain? These two files were added to fill the gaps in the other sources: `epl-00-25` stops 30 matches before the end of 2024/25, and `epl-25-26` only has results for matchweeks 1–3 of 2025/26. We look at their structure, columns and coverage, run internal consistency checks, and compare them with the older sources wherever they overlap.

Both are [football-data.co.uk](https://www.football-data.co.uk/englandm.php) season files in that site's original format: abbreviated column names (`FTHG`, `HST`, …), `dd/mm/yyyy` dates, and the same short team names as `epl-00-25` (`Man United`, `Nott'm Forest`). They sit at the top of `raw_data/` rather than in their own dataset folder.

This notebook is **read-only**: it never modifies `raw_data/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Resolve the repo root whether the notebook runs from notebooks/ or the root
ROOT = Path.cwd() if (Path.cwd() / "raw_data").exists() else Path.cwd().parent
RAW = ROOT / "raw_data"
FILES = {"2024-25": "2425.csv", "2025-26": "2526.csv"}
sorted(p.name for p in RAW.iterdir() if p.is_file())

['2425.csv', '2526.csv']

## 1. Load

In [2]:
# The files start with a UTF-8 byte-order mark, so read with utf-8-sig or the first column is named '\ufeffDiv'
raw = {season: pd.read_csv(RAW / name, encoding="utf-8-sig") for season, name in FILES.items()}
for season, df in raw.items():
    df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
    print(f"{FILES[season]}: shape {df.shape}  |  file size {(RAW / FILES[season]).stat().st_size / 1024:.0f} KB")
raw["2025-26"].iloc[:5, :24]

2425.csv: shape (380, 120)  |  file size 192 KB
2526.csv: shape (380, 132)  |  file size 199 KB


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,E0,2025-08-15,20:00,Liverpool,Bournemouth,4,2,H,1,0,H,A Taylor,19,10,10,3,7,10,6,7,1,2,0,0
1,E0,2025-08-16,12:30,Aston Villa,Newcastle,0,0,D,0,0,D,C Pawson,3,16,3,3,13,11,3,6,1,1,1,0
2,E0,2025-08-16,15:00,Brighton,Fulham,1,1,D,0,0,D,S Barrott,10,7,4,2,16,15,4,3,3,3,0,0
3,E0,2025-08-16,15:00,Sunderland,West Ham,3,0,H,0,0,D,R Jones,10,12,5,4,8,10,5,7,0,1,0,0
4,E0,2025-08-16,15:00,Tottenham,Burnley,3,0,H,1,0,H,M Oliver,16,14,6,4,14,8,6,5,0,0,0,0


## 2. Column dictionary

The first 24 columns are the same in both files. Every stat comes as a home/away pair.

| Column | Meaning |
|---|---|
| `Div` | Division code (`E0` = Premier League) |
| `Date`, `Time` | Match date (`dd/mm/yyyy`) and kick-off time (UK) |
| `HomeTeam`, `AwayTeam` | Teams (football-data.co.uk short names) |
| `FTHG`, `FTAG`, `FTR` | Full-time goals and result (`H` / `D` / `A`) |
| `HTHG`, `HTAG`, `HTR` | Half-time goals and result |
| `Referee` | Match referee |
| `HS`, `AS` / `HST`, `AST` | Shots / shots on target |
| `HF`, `AF` | Fouls committed |
| `HC`, `AC` | Corners |
| `HY`, `AY` / `HR`, `AR` | Yellow / red cards |

Everything after that is **betting odds**: pre-match (`B365H`, `PSD`, …) and closing (`B365CH`, …) 1X2 prices per bookmaker, market maximum and average (`Max*`, `Avg*`), over/under 2.5 goals, and Asian handicap lines. The set of bookmakers changes between the two seasons, so the odds columns differ:

In [3]:
CORE = list(raw["2024-25"].columns[:24])
assert list(raw["2025-26"].columns[:24]) == CORE, "the match columns should be identical in both files"

only_2425 = sorted(set(raw["2024-25"].columns) - set(raw["2025-26"].columns))
only_2526 = sorted(set(raw["2025-26"].columns) - set(raw["2024-25"].columns))
print(f"odds columns: 2024-25 {raw['2024-25'].shape[1] - 24}, 2025-26 {raw['2025-26'].shape[1] - 24}")
print(f"only in 2425.csv ({len(only_2425)}): {only_2425}")
print(f"only in 2526.csv ({len(only_2526)}): {only_2526}")

odds columns: 2024-25 96, 2025-26 108
only in 2425.csv (18): ['1XBA', '1XBCA', '1XBCD', '1XBCH', '1XBD', '1XBH', 'BFA', 'BFCA', 'BFCD', 'BFCH', 'BFD', 'BFH', 'WHA', 'WHCA', 'WHCD', 'WHCH', 'WHD', 'WHH']
only in 2526.csv (30): ['BFDA', 'BFDCA', 'BFDCD', 'BFDCH', 'BFDD', 'BFDH', 'BMGMA', 'BMGMCA', 'BMGMCD', 'BMGMCH', 'BMGMD', 'BMGMH', 'BVA', 'BVCA', 'BVCD', 'BVCH', 'BVD', 'BVH', 'CLA', 'CLCA', 'CLCD', 'CLCH', 'CLD', 'CLH', 'LBA', 'LBCA', 'LBCD', 'LBCH', 'LBD', 'LBH']


## 3. Types, missing values and ranges

In [4]:
for season, df in raw.items():
    core_nulls = df[CORE].isna().sum()
    odds_nulls = df.iloc[:, 24:].isna().sum()
    print(f"{season}: missing values in match columns: {int(core_nulls.sum())}  |  "
          f"odds columns with gaps: {int((odds_nulls > 0).sum())} of {len(odds_nulls)} (worst: {odds_nulls.idxmax()}, {odds_nulls.max()} missing)")

raw["2025-26"][CORE].describe().T[["min", "max", "mean"]].round(2)

2024-25: missing values in match columns: 0  |  odds columns with gaps: 24 of 96 (worst: BWH, 141 missing)
2025-26: missing values in match columns: 0  |  odds columns with gaps: 56 of 108 (worst: PSH, 170 missing)


,min,max,mean
Date,2025-08-15 00:00:00,2026-05-24 00:00:00,2026-01-04 07:15:47.368421
FTHG,0.0,5.0,1.526316
FTAG,0.0,5.0,1.223684
HTHG,0.0,3.0,0.692105
HTAG,0.0,4.0,0.497368
HS,1.0,35.0,13.839474
AS,3.0,30.0,11.157895
HST,0.0,11.0,4.502632
AST,0.0,11.0,3.871053
HF,3.0,20.0,10.652632


## 4. Season coverage

A complete 20-team season has **380 matches**: every team plays 38, and every pair meets once at each ground.

In [5]:
rows = []
for season, df in raw.items():
    appearances = pd.concat([df["HomeTeam"], df["AwayTeam"]]).value_counts()
    rows.append({
        "season": season, "matches": len(df), "teams": appearances.size,
        "games_per_team": f"{appearances.min()}-{appearances.max()}",
        "duplicate_fixtures": int(df.duplicated(["HomeTeam", "AwayTeam"]).sum()),
        "first": df["Date"].min().date(), "last": df["Date"].max().date(),
        "referees": df["Referee"].nunique(),
    })
coverage = pd.DataFrame(rows).set_index("season")
coverage

,matches,teams,games_per_team,duplicate_fixtures,first,last,referees
season,,,,,,,
2024-25,380,20,38-38,0,2024-08-16,2025-05-25,26
2025-26,380,20,38-38,0,2025-08-15,2026-05-24,23


In [6]:
# Test: both seasons are complete
assert (coverage["matches"] == 380).all() and (coverage["teams"] == 20).all()
assert (coverage["games_per_team"] == "38-38").all() and (coverage["duplicate_fixtures"] == 0).all()

promoted = sorted(set(raw["2025-26"]["HomeTeam"]) - set(raw["2024-25"]["HomeTeam"]))
relegated = sorted(set(raw["2024-25"]["HomeTeam"]) - set(raw["2025-26"]["HomeTeam"]))
print(f"relegated after 2024-25: {relegated}  |  promoted for 2025-26: {promoted}")

relegated after 2024-25: ['Ipswich', 'Leicester', 'Southampton']  |  promoted for 2025-26: ['Burnley', 'Leeds', 'Sunderland']


## 5. Consistency checks

In [7]:
def result(home, away):
    return pd.Series("D", index=home.index).mask(home > away, "H").mask(home < away, "A")

checks = {}
for season, df in raw.items():
    checks[season] = {
        "FTR matches the full-time goals": (result(df["FTHG"], df["FTAG"]) == df["FTR"]).all(),
        "HTR matches the half-time goals": (result(df["HTHG"], df["HTAG"]) == df["HTR"]).all(),
        "half-time goals <= full-time goals": ((df["HTHG"] <= df["FTHG"]) & (df["HTAG"] <= df["FTAG"])).all(),
        "shots on target <= shots": ((df["HST"] <= df["HS"]) & (df["AST"] <= df["AS"])).all(),
        "only Premier League rows (Div == E0)": (df["Div"] == "E0").all(),
        "every date falls inside one Aug-May season": df["Date"].dt.year.where(df["Date"].dt.month >= 7, df["Date"].dt.year - 1).nunique() == 1,
    }
checks = pd.DataFrame(checks)
assert checks.all().all()
checks

,2024-25,2025-26
FTR matches the full-time goals,True,True
HTR matches the half-time goals,True,True
half-time goals <= full-time goals,True,True
shots on target <= shots,True,True
only Premier League rows (Div == E0),True,True
every date falls inside one Aug-May season,True,True


## 6. Overlap with the older sources

Clean data already had part of each season: 350 matches of 2024/25 from `epl-00-25`, and 30 matches of 2025/26 from `epl-25-26`. Where a match is in both, do the scores, dates and (for `epl-00-25`) match stats agree?

In [8]:
sys.path.append(str(ROOT / "scripts"))
from build_clean_data import TEAM_LOOKUP  # the canonical team-name mapping used to build clean_data/

unmapped = sorted(set(pd.concat([df[["HomeTeam", "AwayTeam"]].stack() for df in raw.values()])) - TEAM_LOOKUP.keys())
print("team names not yet mapped in build_clean_data.TEAMS:", unmapped or "none")

def keyed(df, home, away):
    df = df.copy()  # the wide odds tables are fragmented; copy before adding columns
    df["home"], df["away"] = df[home].map(TEAM_LOOKUP), df[away].map(TEAM_LOOKUP)
    return df.set_index(["home", "away"])

# 2024-25: compare with epl-00-25
old00 = pd.read_csv(RAW / "epl-00-25" / "epl_final.csv", parse_dates=["MatchDate"]).query("Season == '2024/25'")
new24, old24 = keyed(raw["2024-25"], "HomeTeam", "AwayTeam"), keyed(old00, "HomeTeam", "AwayTeam")
both24 = new24.join(old24, how="inner", rsuffix="_old")
pairs = {"date": ("Date", "MatchDate"), "score": ("FTHG", "FullTimeHomeGoals"), "away score": ("FTAG", "FullTimeAwayGoals"),
         "shots": ("HS", "HomeShots"), "shots on target": ("AST", "AwayShotsOnTarget"), "corners": ("HC", "HomeCorners"),
         "fouls": ("AF", "AwayFouls"), "yellow cards": ("HY", "HomeYellowCards"), "red cards": ("AR", "AwayRedCards")}
print(f"\n2024-25: {len(both24)} matches in both 2425.csv and epl-00-25; {len(new24) - len(both24)} only in 2425.csv")
for label, (new_col, old_col) in pairs.items():
    print(f"  {label:16s} disagreements: {int((both24[new_col] != both24[old_col]).sum())}")
only24 = new24.index.difference(old24.index)
print(f"  matches only in 2425.csv run from {new24.loc[only24, 'Date'].min().date()} to {new24.loc[only24, 'Date'].max().date()}")

team names not yet mapped in build_clean_data.TEAMS: none

2024-25: 350 matches in both 2425.csv and epl-00-25; 30 only in 2425.csv
  date             disagreements: 0
  score            disagreements: 0
  away score       disagreements: 0
  shots            disagreements: 0
  shots on target  disagreements: 0
  corners          disagreements: 0
  fouls            disagreements: 0
  yellow cards     disagreements: 0
  red cards        disagreements: 0
  matches only in 2425.csv run from 2025-05-10 to 2025-05-25


In [9]:
# 2025-26: compare with the played matches in epl-25-26 (FBref, one row per team per match)
fbref = pd.read_csv(RAW / "epl-25-26" / "team_match_stats.csv", parse_dates=["date"])
fbref_home = fbref[fbref["result"].notna() & (fbref["venue"] == "Home")]
new25, old25 = keyed(raw["2025-26"], "HomeTeam", "AwayTeam"), keyed(fbref_home, "team", "opponent")
both25 = new25.join(old25, how="inner", rsuffix="_old")
print(f"2025-26: {len(both25)} matches in both 2526.csv and epl-25-26; {len(new25) - len(both25)} only in 2526.csv")
print(f"  date disagreements:  {int((both25['Date'] != both25['date']).sum())}")
print(f"  score disagreements: {int(((both25['FTHG'] != both25['GF']) | (both25['FTAG'] != both25['GA'])).sum())}")

2025-26: 30 matches in both 2526.csv and epl-25-26; 350 only in 2526.csv
  date disagreements:  0
  score disagreements: 0


In [10]:
# Test: no conflicts, and the new files cover exactly what was missing
assert not unmapped
assert all((both24[n] == both24[o]).all() for n, o in pairs.values())
assert (both25["Date"] == both25["date"]).all() and (both25["FTHG"] == both25["GF"]).all() and (both25["FTAG"] == both25["GA"]).all()
assert len(both24) == len(old24) == 350 and len(both25) == len(old25) == 30
print("overlap checks: OK")

overlap checks: OK


## 7. Summary

| | `2425.csv` | `2526.csv` |
|---|---|---|
| **Grain** | One row per match (home perspective), 380 rows × 120 columns | 380 rows × 132 columns |
| **Span** | 2024/25, 2024-08-16 to 2025-05-25 | 2025/26, 2025-08-15 to 2026-05-24 |
| **Coverage** | Complete: 20 teams × 38 games | Complete: 20 teams × 38 games |
| **Nulls** | None in the 24 match columns; gaps only in some bookmakers' odds | Same |

**Things to keep in mind downstream**
- **These files complete the two seasons the other sources left unfinished.** `2425.csv` adds the last 30 matches of 2024/25 (10–25 May 2025) that `epl-00-25` is missing. `2526.csv` adds the 350 matches of 2025/26 after matchweek 3 that `epl-25-26` lacks. `scripts/build_clean_data.py` uses them for exactly that.
- **They agree with the older sources on every overlapping match**: all 350 shared 2024/25 matches match `epl-00-25` on date, score and every match stat checked, and all 30 shared 2025/26 matches match `epl-25-26` on date and score.
- **Read with `encoding="utf-8-sig"`** (the files start with a byte-order mark) and parse `Date` with `format="%d/%m/%Y"`.
- **Column names are football-data.co.uk abbreviations** (`FTHG`, `HST`, …), not the spelled-out names used in `epl-00-25`. The first 24 columns are identical in both files; the betting-odds columns after them differ because the bookmakers covered changed between seasons.
- **Team names are the short football-data.co.uk names**, the same as `epl-00-25`, and are already mapped in `build_clean_data.TEAMS`.
- **Unlike the older sources, they include `Referee`, kick-off `Time` and betting odds.** Those only exist for these two seasons (plus `epl-25-26` for referee and time in 2025/26), so they stay out of `clean_data/` under the every-season rule.
- The files sit at the top of `raw_data/` instead of in a dataset folder like the others. If they're moved into one, update the `SRC_FD_*` paths in `build_clean_data.py` and `RAW` / `FILES` at the top of this notebook.
